# Train - CDC Diabetes Health Indicators

Trains a Neural Network (MLP) on `Dataset/processed/diabetes_indicators_clean.csv` to predict diabetes status, and saves plots/metrics to `Code/outputs/diabetes_indicators/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    average_precision_score, balanced_accuracy_score
)

FIG = "outputs/diabetes_indicators"
RES = "outputs/diabetes_indicators" 

In [ ]:
df = pd.read_csv("../Dataset/processed/diabetes_indicators_clean.csv")

y = df["Diabetes_binary"].astype(int)
X = df.drop(columns=["Diabetes_binary"])
print("Target balance:")
print(y.value_counts(normalize=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32), activation="relu", alpha=1e-4,
    early_stopping=True, n_iter_no_change=10, max_iter=200,
    random_state=42
)
mlp.fit(X_train_s, y_train)

y_pred = mlp.predict(X_test_s)
y_prob = mlp.predict_proba(X_test_s)[:, 1]

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))
print("Balanced accuracy:", round(bal_acc, 4))
print("Training iterations run:", mlp.n_iter_)

In [ ]:
with open(f"{RES}/nn_diabetes_metrics.json", "w") as f:
    json.dump({
        "classification_report": report,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "balanced_accuracy": bal_acc,
        "n_train": len(X_train), "n_test": len(X_test),
        "n_features": X.shape[1],
        "positive_rate_test": float(y_test.mean()),
        "n_iter": int(mlp.n_iter_),
    }, f, indent=2)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["No diabetes", "Pre/diabetes"],
            yticklabels=["No diabetes", "Pre/diabetes"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Neural Network: confusion matrix\n(CDC lifestyle survey, default 0.5 threshold)")
plt.tight_layout()
plt.savefig(f"{FIG}/nn_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"Neural Network (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve: diabetes status model")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG}/nn_roc_curve.png", dpi=150)
plt.show()

In [ ]:
# MLPs have no built-in feature importance, so permutation importance on a subsample for speed
sample_idx = np.random.RandomState(42).choice(len(X_test_s), size=min(8000, len(X_test_s)), replace=False)
perm = permutation_importance(
    mlp, X_test_s[sample_idx], y_test.iloc[sample_idx],
    n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1
)
importances = pd.Series(perm.importances_mean, index=X.columns).nlargest(15)
plt.figure(figsize=(7, 6))
importances.sort_values().plot(kind="barh", color="seagreen")
plt.xlabel("Permutation importance (drop in ROC-AUC)")
plt.title("Neural Network: top 15 features\n(CDC lifestyle survey)")
plt.tight_layout()
plt.savefig(f"{FIG}/nn_feature_importance.png", dpi=150)
plt.show()

importances.to_csv(f"{RES}/nn_feature_importance.csv", header=["importance"])
importances.sort_values(ascending=False).head(10)

In [ ]:
# default 0.5 cutoff looks fine on precision but recall is weak given the class imbalance,
# sweep thresholds and report the F1-optimal one alongside the default
from sklearn.metrics import precision_recall_curve, f1_score

thresholds = np.linspace(0.05, 0.95, 37)
f1s = [f1_score(y_test, (y_prob >= t).astype(int)) for t in thresholds]
best_t = thresholds[int(np.argmax(f1s))]
best_pred = (y_prob >= best_t).astype(int)

from sklearn.metrics import precision_score, recall_score

threshold_result = {
    "default_threshold_0.5": {
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    },
    "f1_optimal_threshold": float(best_t),
    "at_optimal_threshold": {
        "precision": precision_score(y_test, best_pred),
        "recall": recall_score(y_test, best_pred),
        "f1": f1_score(y_test, best_pred),
    },
}
with open(f"{RES}/nn_threshold_analysis.json", "w") as f:
    json.dump(threshold_result, f, indent=2)

threshold_result